# 15. Decision trees and rule-like decisions

**Level:** Intermediate

**Prerequisites:** 11 and 14: classification and evaluation.

[Course index](00_START_HERE.ipynb) | [Previous](14_Logistic_Regression.ipynb) | [Next](16_Random_Forests.ipynb)

**Topics:** Decision trees, Gini impurity, Depth, Pruning, Interpretability.

**Matching script:** `examples\15_Decision_Trees.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

A decision tree recursively partitions feature space using threshold tests. Leaves predict a
class or numerical value. Splits are chosen to improve a criterion such as Gini impurity for
classification or squared-error reduction for regression.

Trees can model nonlinear interactions without scaling numeric features. Deep unrestricted
trees can memorize training data and react strongly to small changes in data. max_depth,
min_samples_leaf, and pruning control complexity. A simple-looking rule is not automatically
a trustworthy causal explanation; a split may reflect a confounder or leakage.

Feature importance from impurity reductions can favor features with more candidate split
points. For interpretation, compare held-out permutation importance and inspect behavior
across groups. Always test on separate data rather than reporting the tree's training fit alone.

## Mathematics or data contract

Binary Gini impurity is 1 - p^2 - (1-p)^2; a pure leaf has impurity zero.

## Important limitation

Readable exported rules describe this fitted model, not guaranteed real-world engineering rules.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Train a shallow tree and inspect its rules

**Purpose:** Connect a fitted model to its branching structure.

### Read the code in this order
1. Generate informative synthetic data.
2. Keep the test partition untouched during fitting.
3. Limit depth and leaf size.
4. Print a short rule listing and check basic held-out performance.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
held-out accuracy: 0.973
|--- feature_1 <= 0.29
|   |--- feature_0 <= -0.86
|   |   |--- truncated branch of depth 2
|   |--- feature_0 >  -0.86
|   |   |--- truncated branch of depth 2
|--- feature_1 >  0.29
|   |--- feature_3 <= 0.60
|   |   |--- class: 1
|   |--- feature_3 >  0.60
|   |   |--- truncated branch of depth 2
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

X, y = make_classification(n_samples=300, n_features=4, n_informative=3,
                           n_redundant=0, class_sep=1.6, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=42)
model.fit(X_train, y_train)
score = accuracy_score(y_test, model.predict(X_test))
assert model.get_depth() <= 3 and score > 0.65
print("held-out accuracy:", round(score, 3))
print(export_text(model, max_depth=1).strip())

held-out accuracy: 0.973
|--- feature_1 <= 0.29
|   |--- feature_0 <= -0.86
|   |   |--- truncated branch of depth 2
|   |--- feature_0 >  -0.86
|   |   |--- truncated branch of depth 2
|--- feature_1 >  0.29
|   |--- feature_3 <= 0.60
|   |   |--- class: 1
|   |--- feature_3 >  0.60
|   |   |--- truncated branch of depth 2


### Interpret example 1

The result should demonstrate this purpose: **Connect a fitted model to its branching structure.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Compare depths 1, 3, and unrestricted using validation data; keep the final test set untouched.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Decision trees` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: Binary Gini impurity is 1 - p^2 - (1-p)^2; a pure leaf has impurity zero.
- Name the boundary of the demonstration: Readable exported rules describe this fitted model, not guaranteed real-world engineering rules.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.